In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Load cleaned datasets
df = pd.read_csv("../Data/Cleaned/ChargingRecords_Cleaned.csv")

# Display first five records
display(df.head())

In [ ]:
# Convert StartDatetime into datetime format
df["StartDatetime"] = pd.to_datetime(df["StartDatetime"])

# Extracting date and time feature


df['StartHour'] = df['StartDatetime'].dt.hour
df['DayOfWeek'] = df['StartDatetime'].dt.dayofweek
df['Month'] = df['StartDatetime'].dt.month
df['DayOfMonth'] = df['StartDatetime'].dt.day

# Weekend Indicator (Saturday = 5, Sunday = 6)
df['IsWeekend'] =(df['DayOfWeek'].isin([5,6]).astype(int))

 # Display Feature
 
display(df[[
    "StartDatetime",
    "StartHour",
    "DayOfWeek",
    "Month",
    "DayOfMonth",
    "IsWeekend"
]].head())

In [ ]:
# Creating a group of 5 categories 

# Defining time category
bins =[0,6,12,17,21,24]

labels=[
    "Night",
    "Morning",
    "Afternoon",
    "Evening",
    "Late Night"
]

# Creating Time of Day feature

df["TimeOfDay"] = pd.cut(
    df['StartHour'],
    bins=bins,
    labels=labels,
    right=False
)

# Check results
display(df[["StartHour","TimeOfDay"]].head(10))

# Check category distribution
display(df["TimeOfDay"].value_counts().sort_index())

In [ ]:
# Creating cyclical Time feature

# Cyclical encoding for hour
df['HourSin'] = np.sin(2 * np.pi * df['StartHour'] / 24)
df['HourCos'] = np.cos(2 * np.pi * df['StartHour'] / 24)

# Cyclical encoding for day of week
df["DaySin"] = np.sin(2 * np.pi * df["DayOfWeek"] / 7)
df["DayCos"] = np.cos(2 * np.pi * df["DayOfWeek"] / 7)

# Cyclical encoding for month
df["MonthSin"] = np.sin(2 * np.pi * (df["Month"] - 1) / 12)
df["MonthCos"] = np.cos(2 * np.pi * (df["Month"] - 1) / 12)

# Verify new features
display(
    df[[
        "StartHour", "HourSin", "HourCos",
        "DayOfWeek", "DaySin", "DayCos",
        "Month", "MonthSin", "MonthCos"
    ]].head()
)

# These features help models understand that December and January, 
# Sunday and Monday, and 11 PM and midnight are neighboring points in a cycle.

In [ ]:
# Identify categorical features

categorical_columns = [
    "Location",
    "ChargerType",
    "ChargerCompany",
    "TimeOfDay"
]

# Check unique values and counts
for column in categorical_columns:
    print(f"\n{column}")
    print("Unique values:", df[column].nunique())
    print(df[column].value_counts())

In [ ]:
# Encode categorical features

# One-Hot Encoding
df_encoded = pd.get_dummies(
    df,
    columns=["Location", "TimeOfDay"],
    dtype=int
)

# Check encoded columns
encoded_columns = [
    col for col in df_encoded.columns
    if col.startswith(("Location_", "TimeOfDay_"))
]

print("Number of encoded columns:", len(encoded_columns))
print("\nEncoded columns:")
print(encoded_columns)

display(df_encoded[encoded_columns].head())

In [ ]:
# Selecting final features and target

# Define features for prediction
feature_columns = [
    "ChargerType",
    "ChargerCompany",
    "IsWeekend",
    "DayOfMonth",
    "HourSin",
    "HourCos",
    "DaySin",
    "DayCos",
    "MonthSin",
    "MonthCos"
]

# Add one-hot encoded Location features
feature_columns += [
    col for col in df_encoded.columns
    if col.startswith("Location_")
]

# Prepare X and y
X = df_encoded[feature_columns].copy()
y = df_encoded["Demand"].copy()

# Check dataset
print("Feature shape:", X.shape)
print("Target shape:", y.shape)

print("\nMissing values in features:", X.isnull().sum().sum())
print("Missing values in target:", y.isnull().sum())

display(X.head())

In [ ]:
# Combine features and target for correlation analysis
correlation_data = X.copy()
correlation_data["Demand"] = y

# Correlation with target
demand_correlation = (
    correlation_data.corr(numeric_only=True)["Demand"]
    .drop("Demand")
    .sort_values(key=abs, ascending=False)
)

print("Feature correlation with Demand:")
display(demand_correlation.to_frame())

# Visualize feature correlations
plt.figure(figsize=(18, 12))

sns.heatmap(
    X.corr(),
    cmap="coolwarm",
    center=0,
    annot=False
)

plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

In [14]:
# Calculate feature-to-feature correlations
feature_corr = X.corr().abs()

# Extract upper triangle to avoid duplicate pairs
upper_triangle = feature_corr.where(
    np.triu(np.ones(feature_corr.shape), k=1).astype(bool)
)

# Find pairs with correlation greater than 0.5
high_corr_pairs = (
    upper_triangle.stack()
    .reset_index()
)

high_corr_pairs.columns = [
    "Feature 1",
    "Feature 2",
    "Correlation"
]

high_corr_pairs = high_corr_pairs[
    high_corr_pairs["Correlation"] > 0.5
].sort_values(by="Correlation", ascending=False)

display(high_corr_pairs)

,Feature 1,Feature 2,Correlation
54,IsWeekend,DaySin,0.771610
1,ChargerType,ChargerCompany,0.659472


In [15]:
# Remove IsWeekend from the feature set
X_final = X.drop(columns=["IsWeekend"]).copy()

# Keep Demand as target
y_final = y.copy()

# Verify final dataset
print("Final feature shape:", X_final.shape)
print("Final target shape:", y_final.shape)

print("\nFeature columns:")
print(X_final.columns.tolist())

print("\nMissing values:", X_final.isnull().sum().sum())

Final feature shape: (72843, 23)
Final target shape: (72843,)

Feature columns:
['ChargerType', 'ChargerCompany', 'DayOfMonth', 'HourSin', 'HourCos', 'DaySin', 'DayCos', 'MonthSin', 'MonthCos', 'Location_accommodation', 'Location_apartment', 'Location_bus garage', 'Location_camping', 'Location_company', 'Location_golf', 'Location_hotel', 'Location_market', 'Location_public area', 'Location_public institution', 'Location_public parking lot', 'Location_resort', 'Location_restaurant', 'Location_sightseeing']

Missing values: 0


In [17]:
# Combine final features and target
df_feature_engineered = X_final.copy()
df_feature_engineered["Demand"] = y_final

# Final validation
print("Dataset shape:", df_feature_engineered.shape)
print("Duplicate rows:", df_feature_engineered.duplicated().sum())
print("Missing values:", df_feature_engineered.isnull().sum().sum())
print("Infinite values:", np.isinf(
    df_feature_engineered.select_dtypes(include=np.number)
).sum().sum())

# Check target distribution
print("\nDemand statistics:")
display(df_feature_engineered["Demand"].describe())

# Save feature-engineered dataset
df_feature_engineered.to_csv(
    "../Data/Cleaned/ChargingRecords_Feature_Engineered.csv",
    index=False
)

print("\nFeature-engineered dataset saved successfully!")

Dataset shape: (72843, 24)
Duplicate rows: 203
Missing values: 0
Infinite values: 0

Demand statistics:


count    72843.000000
mean        17.442667
std         13.491446
min          0.010000
25%          7.530000
50%         14.100000
75%         23.200000
max         97.000000
Name: Demand, dtype: float64


Feature-engineered dataset saved successfully!
